# Temporal Entity Consistency Evaluation

This notebook computes **Mean Track Continuity (MTC)** and **Object Count Consistency (OCC)** directly from the JSON files produced by `data_pipeline/jupiter_notebooks/obj_det_w_dino.ipynb`. It accepts either the `video_general_obj_det_finished.zip` archive or the extracted `video_general_obj_det_finished` directory. ZIP input is extracted automatically to a temporary directory. The notebook does not read video frames or use confidence scores.

The metric definitions are documented in `docs/temporal_entity_consistency_spec.md`.

## Configuration

`OBJECT_DETECTION_OUTPUT` must point either to `video_general_obj_det_finished.zip` or to the extracted `video_general_obj_det_finished` directory created by `obj_det_w_dino.ipynb`. Directory discovery is recursive because that notebook writes each result as `<video>/<video>.json`. Files that do not contain a `detected_objects` mapping are skipped.

In [ ]:
from pathlib import Path

# ZIP produced by obj_det_w_dino.ipynb and stored on Google Drive.
DRIVE_INPUT_ZIP = Path('/content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT/video_general_obj_det_finished.zip')
# The notebook copies the ZIP here before extracting and evaluating it.
LOCAL_INPUT_ZIP = Path('/content/video_general_obj_det_finished.zip')
OBJECT_DETECTION_OUTPUT = LOCAL_INPUT_ZIP

# To evaluate an already local archive or extracted directory instead, set this to False
# and update OBJECT_DETECTION_OUTPUT accordingly.
COPY_INPUT_FROM_DRIVE = True
# A single ZIP archive containing the summary and detailed tables is written to Google Drive.
DRIVE_OUTPUT_ZIP = Path('/content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT/temporal_entity_consistency_metrics.zip')
RECURSIVE = True
SAVE_RESULTS = True

## Mount Google Drive

In [ ]:
import shutil

if COPY_INPUT_FROM_DRIVE or (SAVE_RESULTS and str(DRIVE_OUTPUT_ZIP).startswith('/content/drive/')):
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except ImportError:
        print('Not running in Google Colab; make sure the configured input and output paths are accessible.')

if COPY_INPUT_FROM_DRIVE:
    if not DRIVE_INPUT_ZIP.is_file():
        raise FileNotFoundError(
            f'Object-detection ZIP not found on Google Drive: {DRIVE_INPUT_ZIP}'
        )
    LOCAL_INPUT_ZIP.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_INPUT_ZIP, LOCAL_INPUT_ZIP)
    print(f'Copied input archive to local runtime: {LOCAL_INPUT_ZIP}')

## Metric implementation

In [ ]:
import json
import math
import tempfile
import zipfile
from collections import Counter

import pandas as pd
from IPython.display import display


_temporary_extraction = None


def prepare_input_path(input_path: Path) -> Path:
    global _temporary_extraction
    input_path = Path(input_path).expanduser()
    if input_path.is_file() and input_path.suffix.lower() == '.zip':
        _temporary_extraction = tempfile.TemporaryDirectory(
            prefix='temporal_entity_consistency_'
        )
        extraction_root = Path(_temporary_extraction.name).resolve()
        with zipfile.ZipFile(input_path) as archive:
            for member in archive.infolist():
                destination = (extraction_root / member.filename).resolve()
                if destination != extraction_root and extraction_root not in destination.parents:
                    raise ValueError(f'Unsafe path in ZIP archive: {member.filename}')
            archive.extractall(extraction_root)
        print(f'Extracted {input_path.name} temporarily to {extraction_root}')
        return extraction_root
    return input_path


def discover_json_files(input_path: Path, recursive: bool = True) -> list[Path]:
    input_path = prepare_input_path(input_path)
    if input_path.is_file():
        return [input_path]
    if not input_path.is_dir():
        raise FileNotFoundError(f'Input path does not exist: {input_path}')
    pattern = '**/*.json' if recursive else '*.json'
    return sorted(input_path.glob(pattern))


def load_detection_tracks(path: Path):
    with path.open('r', encoding='utf-8') as source:
        payload = json.load(source)

    raw_tracks = payload.get('detected_objects')
    if not isinstance(raw_tracks, dict):
        return None, ['missing or invalid detected_objects mapping']

    tracks = []
    warnings = []
    for track_key, raw_track in raw_tracks.items():
        if not isinstance(raw_track, dict):
            warnings.append(f'{track_key}: track entry is not an object')
            continue
        class_name = raw_track.get('class_name')
        boxes = raw_track.get('bbox')
        if not isinstance(class_name, str) or not class_name.strip():
            warnings.append(f'{track_key}: missing class_name')
            continue
        if not isinstance(boxes, list):
            warnings.append(f'{track_key}: bbox is not a list')
            continue
        active = [box is not None for box in boxes]
        if not any(active):
            warnings.append(f'{track_key}: no non-null bounding boxes')
            continue
        tracks.append({
            'track_key': str(track_key),
            'track_id': raw_track.get('track_id', track_key),
            'class_name': class_name.strip(),
            'active': active,
        })
    return tracks, warnings


def smallest_mode(values: list[int]) -> int:
    frequencies = Counter(values)
    maximum_frequency = max(frequencies.values())
    return min(value for value, frequency in frequencies.items()
               if frequency == maximum_frequency)


def evaluate_file(path: Path):
    tracks, warnings = load_detection_tracks(path)
    if tracks is None:
        return None, [], [], warnings

    video_id = path.stem
    track_rows = []
    for track in tracks:
        observed_frames = [index for index, is_active in enumerate(track['active']) if is_active]
        first_frame = observed_frames[0]
        last_frame = observed_frames[-1]
        span_frames = last_frame - first_frame + 1
        observation_count = len(observed_frames)
        track_rows.append({
            'source_file': str(path),
            'video_id': video_id,
            'track_key': track['track_key'],
            'track_id': track['track_id'],
            'class_name': track['class_name'],
            'first_frame': first_frame,
            'last_frame': last_frame,
            'observed_frames': observation_count,
            'span_frames': span_frames,
            'track_continuity': observation_count / span_frames,
        })

    frame_count = max((len(track['active']) for track in tracks), default=0)
    classes = sorted({track['class_name'] for track in tracks})
    class_rows = []
    for class_name in classes:
        class_tracks = [track for track in tracks if track['class_name'] == class_name]
        counts = []
        for frame_index in range(frame_count):
            count = sum(
                frame_index < len(track['active']) and track['active'][frame_index]
                for track in class_tracks
            )
            counts.append(count)

        positive_frames = [index for index, count in enumerate(counts) if count > 0]
        first_frame = positive_frames[0]
        last_frame = positive_frames[-1]
        interval_counts = counts[first_frame:last_frame + 1]
        positive_counts = [count for count in interval_counts if count > 0]
        modal_count = smallest_mode(positive_counts)
        consistent_frames = sum(count == modal_count for count in interval_counts)
        evaluated_frames = len(interval_counts)
        class_rows.append({
            'source_file': str(path),
            'video_id': video_id,
            'class_name': class_name,
            'track_count': len(class_tracks),
            'first_frame': first_frame,
            'last_frame': last_frame,
            'modal_positive_count': modal_count,
            'evaluated_frames': evaluated_frames,
            'consistent_frames': consistent_frames,
            'object_count_consistency': consistent_frames / evaluated_frames,
        })

    summary = {
        'source_file': str(path),
        'video_id': video_id,
        'frame_count': frame_count,
        'valid_track_count': len(track_rows),
        'observed_class_count': len(class_rows),
        'mean_track_continuity': (
            sum(row['track_continuity'] for row in track_rows) / len(track_rows)
            if track_rows else math.nan
        ),
        'object_count_consistency': (
            sum(row['object_count_consistency'] for row in class_rows) / len(class_rows)
            if class_rows else math.nan
        ),
        'warning_count': len(warnings),
    }
    return summary, track_rows, class_rows, warnings

## Run the evaluation

In [ ]:
json_files = discover_json_files(OBJECT_DETECTION_OUTPUT, RECURSIVE)
if not json_files:
    raise FileNotFoundError(f'No detection JSON files found in {OBJECT_DETECTION_OUTPUT}')

video_summaries = []
track_results = []
class_results = []
diagnostics = []

for json_path in json_files:
    try:
        summary, tracks, classes, warnings = evaluate_file(json_path)
        if summary is None:
            diagnostics.append({
                'source_file': str(json_path),
                'status': 'skipped',
                'message': '; '.join(warnings),
            })
            continue
        video_summaries.append(summary)
        track_results.extend(tracks)
        class_results.extend(classes)
        diagnostics.extend(
            {'source_file': str(json_path), 'status': 'warning', 'message': warning}
            for warning in warnings
        )
    except (OSError, json.JSONDecodeError, ValueError, TypeError) as error:
        diagnostics.append({
            'source_file': str(json_path),
            'status': 'error',
            'message': str(error),
        })

track_df = pd.DataFrame(track_results)
class_df = pd.DataFrame(class_results)
video_df = pd.DataFrame(video_summaries)
diagnostics_df = pd.DataFrame(diagnostics)

if video_df.empty:
    raise RuntimeError('No valid detection JSON files were evaluated. Inspect diagnostics_df.')

dataset_summary_df = pd.DataFrame([{
    'evaluated_videos': len(video_df),
    'valid_tracks': len(track_df),
    'observed_video_classes': len(class_df),
    'mean_track_continuity': video_df['mean_track_continuity'].mean(),
    'object_count_consistency': video_df['object_count_consistency'].mean(),
}])

display(dataset_summary_df)
display(video_df.sort_values('video_id').reset_index(drop=True))

## Inspect detailed results

In [ ]:
display(track_df.sort_values(['video_id', 'class_name', 'track_key']).reset_index(drop=True))
display(class_df.sort_values(['video_id', 'class_name']).reset_index(drop=True))
if not diagnostics_df.empty:
    display(diagnostics_df)

## Save one metrics ZIP archive to Google Drive

In [ ]:
if SAVE_RESULTS:
    DRIVE_OUTPUT_ZIP.parent.mkdir(parents=True, exist_ok=True)

    dataset_report = pd.DataFrame([{
        'scope': 'dataset',
        'video_id': 'ALL',
        'frame_count': video_df['frame_count'].sum(),
        'valid_track_count': len(track_df),
        'observed_class_count': len(class_df),
        'mean_track_continuity': dataset_summary_df.iloc[0]['mean_track_continuity'],
        'object_count_consistency': dataset_summary_df.iloc[0]['object_count_consistency'],
    }])
    video_report = video_df[[
        'video_id', 'frame_count', 'valid_track_count', 'observed_class_count',
        'mean_track_continuity', 'object_count_consistency'
    ]].copy()
    video_report.insert(0, 'scope', 'video')
    metrics_report_df = pd.concat([dataset_report, video_report], ignore_index=True)

    with tempfile.TemporaryDirectory(prefix='temporal_metrics_output_') as temporary_output:
        temporary_output = Path(temporary_output)
        metrics_report_df.to_csv(temporary_output / 'metrics.csv', index=False)
        track_df.to_csv(temporary_output / 'per_track_continuity.csv', index=False)
        class_df.to_csv(temporary_output / 'per_class_object_count_consistency.csv', index=False)
        diagnostics_df.to_csv(temporary_output / 'diagnostics.csv', index=False)

        with zipfile.ZipFile(DRIVE_OUTPUT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
            for result_file in sorted(temporary_output.iterdir()):
                archive.write(result_file, arcname=result_file.name)

    print(f'Metrics archive saved to: {DRIVE_OUTPUT_ZIP}')
    display(metrics_report_df)